# Data Cleaning Pipeline — From Messy CSV to Analysis-Ready Dataset

**Input:**  `sales_data_dirty.csv` — 615 rows with real-world data quality issues  
**Output:** `sales_data_clean.csv` — validated, standardised, ready for analysis  
**Tools:**  Python · pandas · numpy

---

### Issues identified in the source file

| # | Problem | Example |
|---|---|---|
| 1 | Duplicate rows | 15 full-row duplicates |
| 2 | Inconsistent category labels | `'ELECTRONICS'`, `'Electrnics'`, `'Electronics '` |
| 3 | Inconsistent status labels | `'done'`, `'COMPLETED'`, `'Complete'` |
| 4 | Mixed date formats | `2023-01-15`, `15/01/2023`, `Jan 15 2023` |
| 5 | Leading / trailing whitespace | `'  United States  '` |
| 6 | Mixed quantity types | `2`, `'2.0'`, `'two'` |
| 7 | Revenue outliers | values from `-200` to `+14 000` |
| 8 | Missing values | 123 nulls across 6 columns |

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

def section(title):
    print(f'\n{"─" * 55}')
    print(f'  {title}')
    print(f'{"─" * 55}')

## 2. Load & First Look

In [ ]:
df = pd.read_csv('sales_data_dirty.csv', dtype={'quantity': str})

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(8)

In [ ]:
# ── Full quality audit before any cleaning ──────────────────────────────────
audit = pd.DataFrame({
    'dtype'   : df.dtypes,
    'nulls'   : df.isnull().sum(),
    'null_%'  : (df.isnull().mean() * 100).round(1),
    'unique'  : df.nunique(),
    'sample'  : [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
print(audit.to_string())

## 3. Step-by-Step Cleaning

### Step 1 — Remove duplicate rows

In [ ]:
section('STEP 1 — Duplicate rows')

before = len(df)
full_dups = df.duplicated().sum()
id_dups   = df.duplicated(subset='order_id').sum()
print(f'  Full-row duplicates : {full_dups}')
print(f'  Duplicate order_ids : {id_dups}')

df = df.drop_duplicates()                            # remove full-row duplicates
df = df.drop_duplicates(subset='order_id', keep='first')  # keep first of id dups

print(f'\n  Rows before : {before}')
print(f'  Rows after  : {len(df)}')
print(f'  Removed     : {before - len(df)}')

### Step 2 — Strip whitespace from all text columns

In [ ]:
section('STEP 2 — Whitespace')

str_cols = df.select_dtypes('object').columns.tolist()

# show a few dirty examples before
ws_before = df[df['country'].str.startswith('  ', na=False)]['country'].head(4).tolist()
print('  Before:', ws_before)

for col in str_cols:
    df[col] = df[col].str.strip()

ws_after = df['country'].head(4).tolist()
print('  After :', ws_after)

### Step 3 — Standardise category labels

In [ ]:
section('STEP 3 — Product category labels')

print('  Unique values BEFORE:', sorted(df['product_category'].dropna().unique()))

# canonical mapping: lowercase → clean label
cat_map = {
    'electronics'    : 'Electronics',
    'electrnics'     : 'Electronics',
    'office supplies': 'Office Supplies',
    'office_supplies': 'Office Supplies',
    'office  supplies':'Office Supplies',
    'software'       : 'Software',
    'sofware'        : 'Software',
    'soft ware'      : 'Software',
    'accessories'    : 'Accessories',
    'accessoires'    : 'Accessories',
    'accesories'     : 'Accessories',
    'books'          : 'Books',
    'boks'           : 'Books',
    'book'           : 'Books',
}

df['product_category'] = (
    df['product_category']
    .str.lower()
    .str.strip()
    .map(cat_map)
)

print('  Unique values AFTER :', sorted(df['product_category'].dropna().unique()))
print('  Nulls introduced by bad labels:',
      df['product_category'].isnull().sum(), '(will handle in Step 8)')

### Step 4 — Standardise order status

In [ ]:
section('STEP 4 — Order status labels')

print('  Unique values BEFORE:', sorted(df['order_status'].dropna().unique()))

status_map = {
    'completed'  : 'Completed', 'complete': 'Completed', 'done': 'Completed',
    'refunded'   : 'Refunded',  'refund'  : 'Refunded',
    'pending'    : 'Pending',   'in progress': 'Pending',
    'cancelled'  : 'Cancelled', 'canceled': 'Cancelled',
}

df['order_status'] = (
    df['order_status']
    .str.lower()
    .str.strip()
    .map(status_map)
)

print('  Unique values AFTER :', sorted(df['order_status'].dropna().unique()))

### Step 5 — Parse mixed date formats

In [ ]:
section('STEP 5 — Date parsing')

# show the variety before parsing
print('  Sample raw dates:')
print( df['order_date'].head(12).tolist() )

df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=False, infer_datetime_format=True)

print(f'\n  dtype after  : {df["order_date"].dtype}')
print(f'  Range        : {df["order_date"].min().date()}  →  {df["order_date"].max().date()}')
print(f'  Nulls        : {df["order_date"].isnull().sum()}')

### Step 6 — Fix quantity column (mixed int / float-string / word)

In [ ]:
section('STEP 6 — Quantity type fix')

print('  Unique raw values:', sorted(df['quantity'].dropna().unique(), key=str))

word_to_int = {'one':1, 'two':2, 'three':3, 'four':4, 'five':5}

def parse_qty(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    if val_str in word_to_int:
        return word_to_int[val_str]
    try:
        return int(float(val_str))
    except ValueError:
        return np.nan

df['quantity'] = df['quantity'].apply(parse_qty)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('Int64')

print(f'  dtype after  : {df["quantity"].dtype}')
print(f'  Unique after : {sorted(df["quantity"].dropna().unique().tolist())}')
print(f'  Nulls        : {df["quantity"].isnull().sum()}')

### Step 7 — Detect and handle revenue outliers

In [ ]:
section('STEP 7 — Revenue outliers')

rev = df['revenue'].dropna()
Q1, Q3 = rev.quantile(0.25), rev.quantile(0.75)
IQR    = Q3 - Q1
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

neg_mask  = df['revenue'] < 0
high_mask = df['revenue'] > hi

print(f'  IQR bounds   : [{lo:.2f}, {hi:.2f}]')
print(f'  Negative     : {neg_mask.sum()} rows  →  flagged as data error, set to NaN')
print(f'  High outliers: {high_mask.sum()} rows  →  flagged for review, set to NaN')

# visualise before
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['revenue'].dropna(), bins=50, color='steelblue', alpha=0.8)
axes[0].set_title('Revenue distribution — BEFORE', fontweight='bold')
axes[0].axvline(0,  color='red',   linestyle='--', label='zero')
axes[0].axvline(hi, color='orange',linestyle='--', label=f'IQR upper = {hi:.0f}')
axes[0].legend(fontsize=8)

df.loc[neg_mask | high_mask, 'revenue'] = np.nan

axes[1].hist(df['revenue'].dropna(), bins=50, color='seagreen', alpha=0.8)
axes[1].set_title('Revenue distribution — AFTER', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_outliers.png', bbox_inches='tight')
plt.show()

### Step 8 — Handle remaining missing values

In [ ]:
section('STEP 8 — Missing values')

print('  Nulls per column BEFORE imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

# shipping_cost null = free shipping (order > $100), fill with 0
df['shipping_cost'] = df['shipping_cost'].fillna(0)

# revenue null = can't recover reliably → drop row
df = df.dropna(subset=['revenue'])

# unit_price null → fill with category median
df['unit_price'] = df.groupby('product_category')['unit_price'].transform(
    lambda x: x.fillna(x.median())
)

# product_category null → infer from product_name where possible, else 'Unknown'
name_to_cat = {
    'Wireless Headphones':'Electronics', 'Bluetooth Speaker':'Electronics',
    'USB-C Hub':'Electronics', 'Webcam':'Electronics', 'LED Desk Lamp':'Electronics',
    'Ergonomic Mouse':'Office Supplies', 'Mechanical Keyboard':'Office Supplies',
    'Monitor Stand':'Office Supplies', 'Desk Organizer':'Office Supplies',
    'Notebook Set':'Office Supplies',
    'Project Mgmt License':'Software', 'Antivirus Annual':'Software',
    'Cloud Storage Plan':'Software', 'Design Suite':'Software', 'VPN Subscription':'Software',
    'Phone Case':'Accessories', 'Screen Protector':'Accessories',
    'Cable Organizer':'Accessories', 'Laptop Sleeve':'Accessories', 'Travel Adapter':'Accessories',
    'Data Science Handbook':'Books', 'Excel Mastery':'Books',
    'Python Cookbook':'Books', 'Business Analytics':'Books', 'SQL Guide':'Books',
}
mask = df['product_category'].isnull()
df.loc[mask, 'product_category'] = df.loc[mask, 'product_name'].map(name_to_cat)
df['product_category'] = df['product_category'].fillna('Unknown')

# country null → 'Unknown'
df['country'] = df['country'].fillna('Unknown')

# customer_name null → 'Anonymous'
df['customer_name'] = df['customer_name'].fillna('Anonymous')

print('\n  Nulls per column AFTER imputation:')
remaining = df.isnull().sum()[df.isnull().sum() > 0]
print(remaining.to_string() if len(remaining) else '  ✓ No nulls remaining')

### Step 9 — Enforce correct data types

In [ ]:
section('STEP 9 — Final dtype enforcement')

df['revenue']       = pd.to_numeric(df['revenue'],       errors='coerce').round(2)
df['unit_price']    = pd.to_numeric(df['unit_price'],    errors='coerce').round(2)
df['shipping_cost'] = pd.to_numeric(df['shipping_cost'], errors='coerce').round(2)
df['profit']        = pd.to_numeric(df['profit'],        errors='coerce').round(2)
df['discount']      = pd.to_numeric(df['discount'],      errors='coerce').round(2)

print(df.dtypes.to_string())

## 4. Validation — Before vs After

In [ ]:
df_original = pd.read_csv('sales_data_dirty.csv', dtype={'quantity': str})

print('=' * 55)
print('  CLEANING SUMMARY')
print('=' * 55)
print(f'  Rows            : {len(df_original):>5}  →  {len(df):>5}')
print(f'  Total nulls     : {df_original.isnull().sum().sum():>5}  →  {df.isnull().sum().sum():>5}')
print(f'  Category labels : {df_original["product_category"].nunique():>5}  →  {df["product_category"].nunique():>5}')
print(f'  Status labels   : {df_original["order_status"].nunique():>5}  →  {df["order_status"].nunique():>5}')
print(f'  Negative revenue: {(pd.to_numeric(df_original["revenue"], errors="coerce") < 0).sum():>5}  →  {(df["revenue"] < 0).sum():>5}')
print(f'  Date dtype      :  object  →  {df["order_date"].dtype}')
print(f'  Qty dtype       :  object  →  {df["quantity"].dtype}')
print('=' * 55)

In [ ]:
# ── Visual: null heatmap before vs after ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, title in zip(
    axes,
    [df_original, df],
    ['Missing Values — DIRTY', 'Missing Values — CLEAN']
):
    null_pct = (data.isnull().mean() * 100).sort_values(ascending=False)
    colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in null_pct]
    ax.barh(null_pct.index, null_pct.values, color=colors)
    ax.set_title(title, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('chart_nulls_comparison.png', bbox_inches='tight')
plt.show()

## 5. Export Clean Dataset

In [ ]:
# reorder columns for readability
col_order = [
    'order_id', 'order_date', 'customer_name', 'country',
    'product_category', 'product_name', 'quantity',
    'unit_price', 'discount', 'revenue', 'shipping_cost',
    'profit', 'payment_method', 'order_status'
]
df = df[col_order]
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')

df.to_csv('sales_data_clean.csv', index=False)

print('✓ Saved: sales_data_clean.csv')
print(f'  Shape : {df.shape}')
print(f'  Nulls : {df.isnull().sum().sum()}')
df.head(5)